This script combines metrics of storage from the literature with FDC slope calculations and calculates the mean Q5 for the pre-fire period. It exports a csv table with each metric for each watershed which is used to make linear regressions between each metric of storage and mean pre-fire Q5.

In [8]:
#imports
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import os
import math
from scipy.interpolate import interp1d

In [9]:
#set default font to arial
matplotlib.rc('font',family='Arial')

In [15]:
#inputs
working_directory=r"C:\Users\duffshan\Box\Shannon_Duffy\Writing\Code For Paper Publication"
# filepaths
storage_file = os.path.join(working_directory,"Input Data","storage_literature.csv")
BFR_file=os.path.join(working_directory,"Output Data","BFR_intercepts.csv")
lowflowfile = os.path.join(working_directory, "Intermediate Outputs","5thPercentileFlows_nofall_2000_2025.csv")
outputfolder = os.path.join(working_directory, "Intermediate Outputs")
figure_outputs=os.path.join(working_directory, "Output Figures")

In [11]:
#read in storage csv
storage_literature=pd.read_csv(storage_file)

In [22]:
#read in BFR slope
bfr=pd.read_csv(BFR_file)
bfr=bfr[["WATERSHED","BFR_Slope"]]

#merge with storage file
storage_bfr=pd.merge(storage_literature,bfr, on="WATERSHED")

In [12]:
#calculate mean Q5 of pre-fire period
#read in low flows
lowflows = pd.read_csv(lowflowfile)
lowflows=lowflows .rename(columns={"GSLOOK":"LO","GSWS01":"WS 1","GSWS02":"WS 2","GSWS03":"WS 3","GSWS08":"WS 8","GSWS09":"WS 9","GSWS10":"WS 10","GSWSMC":"MC"})

#subset into Holiday Farm Fire and Lookout Fire groups
lowflow_holiday=lowflows[['WATERYEAR',"WS 1","WS 2","WS 3","WS 9","WS 10"]]
lowflow_lookout=lowflows[['WATERYEAR',"LO","WS 8","MC"]]

#select only prefire period
lowflow_holiday = lowflow_holiday[lowflow_holiday["WATERYEAR"]<2021]
lowflow_lookout = lowflow_lookout[lowflow_lookout["WATERYEAR"]<2024]

# find mean and stdev of low flow for each watershed
lowflow_holiday_means = lowflow_holiday.mean()
lowflow_holiday_means=lowflow_holiday_means[1:10]
lowflow_holiday_stdev = lowflow_holiday.std()
lowflow_holiday_stdev=lowflow_holiday_stdev[1:10]

# find mean and stdev of low flow for each watershed
lowflow_lookout_means = lowflow_lookout.mean()
lowflow_lookout_means=lowflow_lookout_means[1:10]
lowflow_lookout_stdev = lowflow_lookout.std()
lowflow_lookout_stdev=lowflow_lookout_stdev[1:10]

#create df of Mean q5 and stdev
q5_holiday_df=pd.DataFrame({"WATERSHED":lowflow_holiday_means.index,"Mean_Q5":lowflow_holiday_means.values,"Q5_stdev":lowflow_holiday_stdev.values})
q5_holiday_df["Q5_2sigma"]=q5_holiday_df["Q5_stdev"]*2
q5_holiday_df["Q5_Confidence"]=q5_holiday_df[["Q5_stdev","WATERSHED"]].apply(lambda x: 1.96*(x["Q5_stdev"]/np.sqrt(21)),axis=1)

q5_lookout_df=pd.DataFrame({"WATERSHED":lowflow_lookout_means.index,"Mean_Q5":lowflow_lookout_means.values,"Q5_stdev":lowflow_lookout_stdev.values})
q5_lookout_df["Q5_2sigma"]=q5_lookout_df["Q5_stdev"]*2
q5_lookout_df["Q5_Confidence"]=q5_lookout_df[["Q5_stdev","WATERSHED"]].apply(lambda x: 1.96*(x["Q5_stdev"]/np.sqrt(24)),axis=1)

q5_df=pd.concat([q5_holiday_df,q5_lookout_df])
print(q5_df)

  WATERSHED       Mean_Q5      Q5_stdev     Q5_2sigma  Q5_Confidence
0      WS 1  5.518140e-07  2.565442e-07  5.130884e-07   1.097258e-07
1      WS 2  2.386430e-06  5.687462e-07  1.137492e-06   2.432568e-07
2      WS 3  2.287641e-06  5.347293e-07  1.069459e-06   2.287075e-07
3      WS 9  5.724569e-07  3.589085e-07  7.178171e-07   1.535077e-07
4     WS 10  7.601955e-07  3.054605e-07  6.109210e-07   1.306476e-07
0        LO  4.639319e-06  9.138723e-07  1.827745e-06   3.656251e-07
1      WS 8  7.777830e-07  3.539846e-07  7.079691e-07   1.416233e-07
2        MC  4.339545e-06  6.058927e-07  1.211785e-06   2.424076e-07


In [23]:
#merge storage dataframe with low flows
merge_storage_lowflows=pd.merge(storage_bfr,q5_df, on="WATERSHED")
#drop stdev and 2 sigma fdc
merge_storage_lowflows=merge_storage_lowflows.drop(columns=["Q5_stdev","Q5_2sigma"])
print(merge_storage_lowflows)

  WATERSHED  DR_MCGUIRE  DR_SEGURA  YOUNG_DEPOSITS  MODERATE_DEPOSITS  \
0      WS 1         NaN   0.163560               8                  4   
1      WS 2    0.084691   0.169827               4                 33   
2      WS 3    0.119190        NaN               8                 28   
3      WS 8    0.081607   0.107059               7                 15   
4      WS 9    0.158254        NaN               2                  4   
5     WS 10    0.134125        NaN               0                  7   
6        MC    0.089980   0.126853               4                  6   
7        LO    0.116140   0.110043               6                 12   

   OLD_DEPOSITS  TOTAL_DEPOSITS  LAVA_1  LAVA_2  FDC_SLOPE  BFR_Slope  \
0             0              12       0      12      0.026   1.056910   
1             7              44       0      22      0.020   1.316925   
2             1              37       0      50      0.020   1.390846   
3            22              44       2      93   

In [24]:
#export to csv
output_path = os.path.join(outputfolder, "storage_lowflow_predictors.csv")
merge_storage_lowflows.to_csv(output_path, index=False)